# Product Pricing Model - Inference Example

This notebook demonstrates how to use the trained product pricing model to make predictions on new data.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

## Load the trained model

In [ ]:
# Path to the model
MODEL_PATH = "../models/best_model/product_pricing_model.pkl"

# Load the model
model = joblib.load(MODEL_PATH)
print(f"Model loaded successfully from {MODEL_PATH}")

## Load and prepare test data

In [ ]:
# Path to test data
TEST_DATA_PATH = "../student_resource/dataset/sample_test.csv"

# Load test data
test_df = pd.read_csv(TEST_DATA_PATH)
print(f"Test data loaded: {test_df.shape[0]} rows and {test_df.shape[1]} columns")

# Display sample data
test_df.head()

## Import feature extraction utilities

In [ ]:
# Import our feature extraction module
from src.featurize import FeatureExtractor

# Initialize feature extractor
feature_extractor = FeatureExtractor()
print("Feature extractor initialized")

## Extract features from test data

In [ ]:
# Extract features
X_test = feature_extractor.transform(test_df)
print(f"Features extracted: {X_test.shape[1]} features")

## Make predictions

In [ ]:
# Make predictions
predictions = model.predict(X_test)

# If the model was trained on log-transformed data, reverse the transformation
predictions = np.exp(predictions) - 1

# Add predictions to the test dataframe
test_df['predicted_price'] = predictions

# Display results
test_df[['asin', 'predicted_price']].head(10)

## Save predictions

In [ ]:
# Create output directory if it doesn't exist
os.makedirs("../outputs", exist_ok=True)

# Save predictions to a CSV file
output_path = "../outputs/sample_predictions.csv"
test_df[['asin', 'predicted_price']].to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

## Evaluate predictions (if ground truth is available)

In [ ]:
from src.utils import evaluate_predictions

try:
    # Try to load ground truth data
    ground_truth_path = "../student_resource/dataset/sample_test_out.csv"
    gt_df = pd.read_csv(ground_truth_path)
    
    # Merge predictions with ground truth
    merged_df = pd.merge(test_df[['asin', 'predicted_price']], 
                         gt_df, 
                         on='asin', 
                         how='inner')
    
    if 'price' in merged_df.columns:
        # Calculate metrics
        metrics = evaluate_predictions(merged_df['price'], merged_df['predicted_price'])
        print("\nEvaluation Metrics:")
        for metric_name, value in metrics.items():
            print(f"{metric_name}: {value:.4f}")
    else:
        print("\nPrice column not found in ground truth data.")
except Exception as e:
    print(f"\nCould not evaluate predictions: {e}")